In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict
from sj_utils.evaluator import TimeChecker

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
HYPERPARAMETERS_PATH = "/workspaces/dev/hyperparameters/libri/avg_2_1.yml"

In [ ]:
hyperparameters = SafetyDict({
    "sentence_max_prev_sentence": 1,
    "weighted_and_offset_token_boundary": 7700,
    "duration_filter_z":{
        "default": 2.0,
        "ko": 2.0,
        "en": 3.7,
    },
    "probability_filter":{
        "z":{
            "default": 3.0,
            "ko": 3.0,
            "en": 3.4,
        },
        "min_prob": {
            "default": 1.0,
            "ko": 0.4,
            "en": 0.15
        },
    },
    "selector":{
        "search_range_sc": {
            "default": 24000,
            "ko": 24000,
            "en": 17000,
        },
        "threshold":{
            "default": 0.5,
            "ko": 0.25,
            "en": 0.3
        },
        "padding": {
            "default": 3200,
            "ko": 3200,
            "en": 5100
        },
        "tolerance": {
            "default": 8000,
            "ko": 8000,
            "en": 15000
        }
    },
    "max_overlap_duration": 96000
})

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter= HYPERPARAMETERS_PATH)

In [ ]:
src = Path(SOURCE)

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
def normalize_text(text):
    return normalize_text_only_en(text).upper()

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

    print(f"Processing")
    print(f"\tAudio name: {flac.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")

    completed = []
    param = Param()
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        transcribe_time.start()
        result:Result = token_streamer.process(param)
        transcribe_time.check()
        completed.extend(result.completed)
        param.update(result)
    completed.extend(result.candidate)

    text = " ".join([s.text for s in completed])
    text = normalize_text(text)

    return TRNFormat(id = flac.stem, text = text)

In [ ]:
processed_time.start()
data = search_all_ref_and_hyp(src, transcriber, normalize_text, 2)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}